# Limpieza de fuentes — los 3 datasets de la rama

Cuaderno que **reproduce exactamente** `script/limpiar_datos.py` (trazabilidad código → ejecución → datos limpios).

> ⚠️ No realiza cruces ni minería: solo prepara/verifica los archivos limpios. La integración (`empleos`) ya está aplicada y versionada en `data/cruce/empleos.parquet`.


## Datasets gestionados

| Dataset | Archivo original | Versión limpia | Rol |
|---|---|---|---|
| JobHop v2 | `data/original/JobHop_v2_train.parquet` | `data/limpia/JobHop_v2_train_limpio.parquet` | Base de trayectorias (quién, cuándo, qué ocupación). **Ya limpio** (proceso previo) → **NO se re-limita** |
| ESCO occupations | `data/original/ESCO/occupations_en.csv` | `data/limpia/ESCO/occupations_en_limpio.csv` | Traduce `code` → nombre de ocupación + grupo ISCO-08 |
| ESCO ISCOGroups | `data/original/ESCO/ISCOGroups_en.csv` | `data/limpia/ESCO/ISCOGroups_en_limpio.csv` | Etiqueta y jerarquía del área ocupacional (`code 4d → label + nivel`) |

Solo estos 3 (los que usa la integración). Los demás ESCO fueron descartados del repo.

## Preparación

Localiza la raíz del proyecto y reutiliza las funciones de `limpiar_datos.py` para que este cuaderno haga **exactamente** lo mismo que el script.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROYECTO = Path.cwd()
while not (PROYECTO / "data").exists() and PROYECTO != PROYECTO.parent:
    PROYECTO = PROYECTO.parent
sys.path.insert(0, str(PROYECTO / "script"))
import limpiar_datos as L

print("Raíz del proyecto:", PROYECTO)
print("Origen :", L.ORIGEN)
print("Destino:", L.DESTINO)

## Qué se procesa (detección automática)

`limpiar_datos.py` recorre `data/original/` y limpia cada `csv`/`parquet` salvo los **excluidos**.

In [ ]:
archivos = [
    p
    for p in sorted(L.ORIGEN.rglob("*"))
    if p.is_file()
    and p.suffix.lower() in (L.EXT_CSV, L.EXT_PARQUET)
    and p.name not in L.ARCHIVOS_EXCLUIDOS
]
display(pd.DataFrame({"archivo": [str(p.relative_to(L.ORIGEN)) for p in archivos]}))
print("Excluido (derivado previo, ya limpio):", L.ARCHIVOS_EXCLUIDOS)

## Procesamiento (mismos pasos y validación que `limpiar_datos.py`)

La función `procesar` aplica: inspección → limpieza (`limpiar_csv`) → validación → guardado en `data/limpia/`.

In [ ]:
def procesar(ruta: Path):
    rel = str(ruta.relative_to(L.ORIGEN)).replace("\\", "/")
    df = L.carga_archivo(ruta)
    L.inspeccionar(
        df, rel, "Parquet" if ruta.suffix.lower() == L.EXT_PARQUET else "CSV"
    )

    cambios = []
    stats = {
        "filas_eliminadas": False,
        "columnas_eliminadas": False,
        "valores_modificados": False,
        "cambio_tipos": False,
        "texto_normalizado": False,
        "duplicados_eliminados": False,
    }
    df_limpio = L.limpiar_csv(df, ruta.stem, cambios, stats)

    validacion = {
        "filas": len(df),
        "columnas": df.shape[1],
        "dups_fila": int(df.duplicated().sum()),
        "nulos_criticos": L.nulos_criticos(df, ruta.stem),
    }
    L.validar(rel, validacion, df_limpio, cambios)

    destino = L.destino_para(ruta.relative_to(L.ORIGEN))
    destino.parent.mkdir(parents=True, exist_ok=True)
    df_limpio.to_csv(destino, index=False, encoding="utf-8")
    print("Guardado en:", destino)
    return df_limpio

### occupations_en.csv

3.043 originales → 3.039: **−4 filas** por `code` duplicado (idénticas salvo `modifiedDate`).

In [ ]:
df_occ = procesar(next(p for p in archivos if p.stem == "occupations_en"))

### ISCOGroups_en.csv

619 originales → 619: se elimina la columna `altLabels` (100 % nula): de 8 a 7 columnas.

In [ ]:
df_isco = procesar(next(p for p in archivos if p.stem == "ISCOGroups_en"))

## JobHop v2 — verificar (no se re-limita)

`JobHop_v2_train.parquet` ya generó su versión limpia en un proceso anterior; aquí solo se **verifica** que exista y se documenta su contenido (no se trata como fuente original).

In [ ]:
ruta_jobhop = PROYECTO / "data" / "limpia" / "JobHop_v2_train_limpio.parquet"
print("Existe la versión limpia:", ruta_jobhop.exists())
df_jobhop = L.carga_archivo(ruta_jobhop)
print("Filas:", len(df_jobhop), "| Columnas:", df_jobhop.shape[1])
display(df_jobhop.head(3))

## Cierre

- Salidas: `data/limpia/ESCO/*_limpio.csv` (este cuaderno) + `data/limpia/JobHop_v2_train_limpio.parquet` (derivado previo, verificado).
- Los originales de `data/original/` **no se modifican**.
- El proceso es **idempotente**: re-ejecutarlo produce los mismos archivos (validación antes/después por archivo).
- Siguiente etapa (fuera de este cuaderno): minería sobre el integrado `data/cruce/empleos.parquet`.